# MNIST Training

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import Input, Flatten, Conv2D, MaxPooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd
import random

2026-06-30 11:33:15.647603: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Data Processing

The dataset is already loaded and split into a training, validation and testing set.

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

ds_train, ds_val, ds_test = tfds.load(
    "cifar10",
    split=[tfds.Split.TRAIN.subsplit(tfds.percent[:80]), 
           tfds.Split.TRAIN.subsplit(tfds.percent[80:]), 
           tfds.Split.TEST],
    as_supervised=True,
    shuffle_files=True,
    batch_size=32
)
normalize = lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.int32))
ds_train = ds_train.map(normalize)
ds_val   = ds_val.map(normalize)
ds_test  = ds_test.map(normalize)

KeyError: "Invalid split train[:80%]. Available splits are: ['test', 'train']"

# TODO: Your Code here

In [ ]:
def build_cnn(num_conv_blocks, num_filters, lr):
    inp = Input(shape=(32, 32, 3))
    x = inp
    
    for _ in range(num_conv_blocks):
        x = Conv2D(num_filters, kernel_size=3, padding="same", activation="relu")(x)
        x = MaxPooling2D(pool_size=2)(x)
    
    x = Flatten()(x)
    x = Dense(128, activation="relu")(x)
    out = Dense(10, activation="softmax")(x)
    
    model = Model(inp, out)
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=tf.keras.optimizers.SGD(learning_rate=lr),
        metrics=["sparse_categorical_accuracy"]
    )
    return model